In [ ]:

import numpy as np
import shutil
import torch
from torch.amp import autocast, GradScaler
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
from tqdm import tqdm
import os
from torchvision.transforms import v2
import json
import matplotlib.pyplot as plt
from torchvision.models import (resnet50, efficientnet_b0,  
                                mobilenet_v3_small, densenet121)
%matplotlib inline

dataset_params = {
        'CIFAR10': {
            'epochs_train': 100,
            'batch_size': 1024,
            'mean_normalize': (0.4914, 0.4822, 0.4465),
            'std_normalize': (0.2023, 0.1994, 0.2010),
            'num_classes': 10,
        },
        'CIFAR100': {
            'epochs_train': 300,
            'batch_size': 512,
            'mean_normalize': (0.5071, 0.4865, 0.4409),
            'std_normalize': (0.2673, 0.2564, 0.2762),
            'num_classes': 100,
        },
        'TinyImageNet':{
            'epochs_train': 300,
            'batch_size': 256,
            'mean_normalize': (0.485, 0.456, 0.406),
            'std_normalize': (0.229, 0.224, 0.225),
            'num_classes': 200,
        },
    }
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
best_acc = 0
best_loss = float('inf')

In [ ]:

def reorganize_tinyimagenet_val_temp(original_val_root, processed_val_root):
    images_dir = os.path.join(original_val_root, 'images')
    annotations_file = os.path.join(original_val_root, 'val_annotations.txt')
    
    if not os.path.exists(annotations_file):
        raise FileNotFoundError(f"Cannot find {annotations_file}")
    if not os.path.exists(images_dir):
        raise FileNotFoundError(f"Cannot find {images_dir}")
    if os.path.exists(processed_val_root):
        subdirs = [d for d in os.listdir(processed_val_root) if os.path.isdir(os.path.join(processed_val_root, d))]
        if len(subdirs) > 0:
            print(f"Using existing processed TinyImageNet val folder: {processed_val_root}")
            return
    print("Creating temporary processed TinyImageNet val folder (this may take a minute)...")
    os.makedirs(processed_val_root, exist_ok=True)
    with open(annotations_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            image_name, class_name = parts[0], parts[1]
            
            class_dir = os.path.join(processed_val_root, class_name)
            os.makedirs(class_dir, exist_ok=True)
            
            src = os.path.join(images_dir, image_name)
            dst = os.path.join(class_dir, image_name)
            
            if os.path.exists(src):
                shutil.copy2(src, dst)
    
    print(f"TinyImageNet val processed and copied to: {processed_val_root}")

def get_data_loaders(dataset_name, dataset_path):
    params = dataset_params[dataset_name]
    mean_normalize = params['mean_normalize']
    std_normalize = params['std_normalize']
    if dataset_name in ['CIFAR10', 'CIFAR100']:
        transform_train = v2.Compose([
            v2.RandomCrop(32, padding=4),
            v2.RandomHorizontalFlip(),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean_normalize, std_normalize),
        ])
    elif dataset_name == 'TinyImageNet':
        transform_train = v2.Compose([
            v2.RandomCrop(64, padding=8),
            v2.RandomHorizontalFlip(),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean_normalize, std_normalize),
        ])
    else:
        raise ValueError(f"Dataset {dataset_name} is not supported.")
    transform_test = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean_normalize, std_normalize),
        ])

    if dataset_name == 'CIFAR10':
        trainset = torchvision.datasets.CIFAR10(root=dataset_path, train=True, download=False, transform=transform_train)
        testset = torchvision.datasets.CIFAR10(root=dataset_path, train=False, download=False, transform=transform_test)
    elif dataset_name == 'CIFAR100':
        trainset = torchvision.datasets.CIFAR100(root=dataset_path, train=True, download=False, transform=transform_train)
        testset = torchvision.datasets.CIFAR100(root=dataset_path, train=False, download=False, transform=transform_test)
    elif dataset_name == 'TinyImageNet':
        train_root = os.path.join(dataset_path, 'train')
        original_val_root = os.path.join(dataset_path, 'val')
        processed_val_root = './tinyimagenet_val_processed'
        reorganize_tinyimagenet_val_temp(original_val_root, processed_val_root)
        trainset = torchvision.datasets.ImageFolder(root=train_root, transform=transform_train)
        testset = torchvision.datasets.ImageFolder(root=processed_val_root, transform=transform_test)
    else:
        raise ValueError(f"Dataset {dataset_name} is not supported.")

    trainloader = torch.utils.data.DataLoader(trainset,
        batch_size=params['batch_size'],shuffle=True,
        pin_memory=True,num_workers=4,prefetch_factor=8,
        persistent_workers=True
    )
    testloader = torch.utils.data.DataLoader(testset,
        batch_size=100,shuffle=False,pin_memory=True,
        num_workers=4,persistent_workers=True
    )

    return trainloader, testloader

def create_model_train(model_name, num_classes, dataset_name):
    model_name = model_name.lower()
    dataset_name = dataset_name.lower()
    if model_name == "resnet50":
        net = resnet50(weights=None)
        net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.maxpool = nn.Identity() 
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif model_name == "efficientnetb0":
        net = efficientnet_b0(weights=None)
        net.features[0][0].stride = (1, 1)
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
    elif model_name == "mobilenetv3small":
        net = mobilenet_v3_small(weights=None)
        net.features[0][0].stride = (1, 1)
        net.classifier[3] = nn.Linear(net.classifier[3].in_features, num_classes)
    elif model_name == "densenet121":
        net = densenet121(weights=None)
        net.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.features.pool0 = nn.Identity()
        net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Model {model_name} is not supported.")
    return net


def mixup_data(x, y, alpha=0.4, device='cuda'):
    lam = torch.distributions.Beta(alpha, alpha).sample().to(device)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train(net, trainloader, device, optimizer, criterion, scaler, epoch):
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for data in tqdm(trainloader):   
        inputs, targets = data
        inputs, targets = inputs.to(device), targets.to(device)
        inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha=0.4, device=device)
        optimizer.zero_grad()
        with autocast(device_type='cuda' if str(device).startswith('cuda') else 'cpu', dtype=torch.float16):
            outputs = net(inputs)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += (lam * predicted.eq(targets_a) + (1 - lam) * predicted.eq(targets_b)).sum().item()

    return train_loss / len(trainloader), correct / total


def test(model_name, net, testloader, device, criterion, dataset_name, epoch):
    global best_acc, best_loss
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for data in tqdm(testloader):
            inputs, targets = data
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    avg_loss = test_loss / len(testloader)
    acc = 100.*correct/total

    if avg_loss < best_loss:
        print(f'Saving model at epoch {epoch} with test loss {avg_loss:.4f}')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
            'loss': avg_loss
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        filename = model_name + "_" + dataset_name + "_bestLoss"
        torch.save(state, './checkpoint/' + filename + '.pth')
        best_loss = avg_loss

    return avg_loss, correct/total


def plotLoss(epochs, history, start_epoch=5):
    plt.plot(range(start_epoch, epochs), history['train_loss'][start_epoch:], '-', linewidth=3, label='Train Loss')
    plt.plot(range(start_epoch, epochs), history['test_loss'][start_epoch:], '-', linewidth=3, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()
    plt.show()


def plotAcc(epochs, history, start_epoch=10):
    plt.plot(range(start_epoch, epochs), history['train_acc'][start_epoch:], '-', linewidth=3, label='Train Acc')
    plt.plot(range(start_epoch, epochs), history['test_acc'][start_epoch:], '-', linewidth=3, label='Test Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()
    plt.show()


def net_train(model_name, dataset_name, dataset_path):

    filename = model_name + "_"+ dataset_name
    params = dataset_params[dataset_name]
    epochs = params['epochs_train']

    net = create_model_train(model_name, params['num_classes'], dataset_name)
    device = 'cuda'
    net = net.to(device)
    cudnn.benchmark = True
    criterion = nn.CrossEntropyLoss()
    head_lr = 0.05  
    optimizer = optim.SGD(net.parameters(), lr=head_lr, momentum=0.9,
    weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {
        'train_loss': [],
        'test_loss': [],
        'train_acc': [],
        'test_acc': []
    }
    os.makedirs('logs', exist_ok=True)
    os.makedirs('weights', exist_ok=True)
    trainloader, testloader = get_data_loaders(dataset_name, dataset_path)
    scaler = GradScaler()
    for epoch in range(epochs):
        print('\nEpoch: %d' % epoch)
        train_loss, train_acc = train(net, trainloader, device, optimizer, criterion, scaler, epoch)
        print("Train \tLoss: %.3f | Acc: %.3f" % (train_loss, train_acc))
        test_loss, test_acc = test(model_name, net, testloader, device, criterion, dataset_name, epoch)
        print("Test \tLoss: %.3f | Acc: %.3f" % (test_loss, test_acc))

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        scheduler.step()

    with open(f'logs/{filename}_history.json', 'w') as f:
        json.dump(history, f)
    torch.save(net.state_dict(), "weights/"+filename)
    print(f"Model weights saved to: weights/{filename}")
    
    plotLoss(epochs, history)    
    plotAcc(epochs, history)

def plot_by_json(model_name,dataset_name):
    filename =model_name+"_"+ dataset_name + "_history.json"
    file_path = os.path.join("logs", filename)
    with open(file_path, "r", encoding="utf-8") as f:
        history = json.load(f)
    total_epochs = len(history['train_loss'])
    plotLoss(total_epochs, history, start_epoch=5)
    plotAcc(total_epochs, history, start_epoch=10)


In [ ]:
#"dataset_path"is the file address of the corresponding dataset, you need change it yourself
'''
model:mobilenetv3small, efficientnetb0, resnet50, densenet121
dataset:CIFAR10, CIFAR100, TinyImageNet
download link:
https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz
http://cs231n.stanford.edu/tiny-imagenet-200.zip
'''
#You can change the model_name and dataset_name to train and plot other models and datasets
net_train('resnet50','CIFAR100', "dataset_path")
plot_by_json('resnet50','CIFAR100')